# V1-20: OpenDART 재무 데이터 계약 검증

이 노트북은 기업 식별이 끝난 뒤, OpenDART 재무제표 원본 행을 V1의 표준 재무 항목으로 **안전하게 읽을 수 있는지** 확인한다. 아직 재무 계산, LangChain Tool, LangGraph workflow를 만들지 않는다.

이전 기업 식별 단계와 차이:

```text
이전: 기업명 → corp_code 후보 / 재확인 상태
이번: corp_code + 사업연도 → CFS/OFS 원본 재무 행 → 표준 재무 항목 후보
```

OpenDART의 단일회사 전체 재무제표 API는 `corp_code`, `bsns_year`, `reprt_code`, `fs_div`를 받고, 응답에 `sj_div`, `account_id`, `account_nm`, `thstrm_amount`, `currency`, `rcept_no` 등을 제공한다. [OpenDART 공식 개발가이드](https://opendart.fss.or.kr/guide/detail.do?apiGrpCd=DS003&apiId=2019020)

## 이번 확인 질문

1. 사업보고서 연간(`11011`) CFS와 OFS에 필요한 계정이 실제로 존재하는가?
2. `account_id` 우선, 계정명 보조 방식으로 매출액·영업이익·자산 등 V1 원천 항목을 찾을 수 있는가?
3. API가 실제로 주는 통화·단위 필드·접수번호와 계정 미존재를 결과 계약에 어떻게 남겨야 하는가?

## 이번에 확정하지 않는 것

- CFS가 일부 계정만 없을 때 OFS를 어느 범위에서 대체할지
- 재무비율 계산식과 0·음수 분모 처리
- 금융/비금융 업종 판정
- 사용자 질문에서 기간을 해석하거나 Tool을 선택하는 방식


In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd


In [ ]:
current_dir = Path.cwd().resolve()
if current_dir.name == 'v1':
    project_root = current_dir.parents[1]
elif current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

env_path = project_root / '.env'
if not env_path.exists():
    raise FileNotFoundError(f'{env_path} 파일을 먼저 만들어 주세요.')

for raw_line in env_path.read_text(encoding='utf-8').splitlines():
    line = raw_line.strip()
    if line and not line.startswith('#') and '=' in line:
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip(chr(39) + chr(34)))

if not os.environ.get('OPENDART_API_KEY'):
    raise ValueError('.env에 OPENDART_API_KEY를 설정해 주세요.')

sys.path.insert(0, str(project_root / 'src'))
from dart.client import fetch_financial_rows, find_corp_info

api_key = os.environ['OPENDART_API_KEY']
print(f'Project root: {project_root}')
print(f'OpenDART API key loaded: {bool(api_key)} (값은 출력하지 않음)')

In [ ]:
# 서로 다른 비금융 상장기업 3곳의 동일 사업연도를 확인한다.
# 이 목록은 최종 평가셋이 아니라 계정·CFS/OFS 구조를 읽기 위한 계약 탐색 표본이다.
CASES = [
    {'기업명': '삼성전자', '사업연도': 2024},
    {'기업명': 'SK하이닉스', '사업연도': 2024},
    {'기업명': '현대제철', '사업연도': 2024},
]
REPORT_CODE = '11011'  # 사업보고서
FS_DIVISIONS = ('CFS', 'OFS')

# V1 결과 계약의 원천 항목이다. 이 매핑은 이번 실행에서 검증할 '후보'이며 아직 최종 구현 규칙이 아니다.
ACCOUNT_CONTRACT = {
    '매출액': {'sj_div': {'IS', 'CIS'}, 'ids': {'ifrs-full_Revenue', 'ifrs-full_RevenueFromContractsWithCustomers'}, 'names': {'매출액', '수익(매출액)'}},
    '영업이익': {'sj_div': {'IS', 'CIS'}, 'ids': {'dart_OperatingIncomeLoss'}, 'names': {'영업이익', '영업이익(손실)'}},
    '당기순이익': {'sj_div': {'IS', 'CIS'}, 'ids': {'ifrs-full_ProfitLoss'}, 'names': {'당기순이익', '당기순이익(손실)'}},
    '자산총계': {'sj_div': {'BS'}, 'ids': {'ifrs-full_Assets'}, 'names': {'자산총계'}},
    '부채총계': {'sj_div': {'BS'}, 'ids': {'ifrs-full_Liabilities'}, 'names': {'부채총계'}},
    '자본총계': {'sj_div': {'BS'}, 'ids': {'ifrs-full_Equity'}, 'names': {'자본총계'}},
    '유동자산': {'sj_div': {'BS'}, 'ids': {'ifrs-full_CurrentAssets'}, 'names': {'유동자산'}},
    '유동부채': {'sj_div': {'BS'}, 'ids': {'ifrs-full_CurrentLiabilities'}, 'names': {'유동부채'}},
    '영업활동현금흐름': {'sj_div': {'CF'}, 'ids': {'ifrs-full_CashFlowsFromUsedInOperatingActivities'}, 'names': {'영업활동으로 인한 현금흐름', '영업활동으로인한현금흐름', '영업활동 현금흐름', '영업활동현금흐름'}},
}

pd.DataFrame(CASES), pd.DataFrame({'표준 항목': ACCOUNT_CONTRACT.keys()})

In [ ]:
# raw_rows에는 OpenDART 원본 행을 그대로 보관한다.
# 호출 오류는 숨기지 않고 상태로 기록한다.
raw_rows, fetch_log = {}, []

for case in CASES:
    try:
        corp = find_corp_info(case['기업명'], api_key)
    except Exception as error:
        for fs_div in FS_DIVISIONS:
            fetch_log.append({
                '기업명': case['기업명'], 'corp_code': None, '사업연도': case['사업연도'],
                '재무제표 기준': fs_div, '상태': '기업 식별 오류', '원본 행 수': 0, '오류': f'{type(error).__name__}: {error}',
            })
        continue

    for fs_div in FS_DIVISIONS:
        key = (corp['corp_code'], case['사업연도'], fs_div)
        try:
            rows = fetch_financial_rows(corp['corp_code'], case['사업연도'], api_key, report_code=REPORT_CODE, fs_div=fs_div)
            raw_rows[key] = rows
            fetch_log.append({
                '기업명': corp['corp_name'], 'corp_code': corp['corp_code'], '사업연도': case['사업연도'],
                '재무제표 기준': fs_div, '상태': '성공', '원본 행 수': len(rows), '오류': None,
            })
        except Exception as error:
            raw_rows[key] = []
            fetch_log.append({
                '기업명': corp['corp_name'], 'corp_code': corp['corp_code'], '사업연도': case['사업연도'],
                '재무제표 기준': fs_div, '상태': '오류', '원본 행 수': 0, '오류': str(error),
            })

fetch_frame = pd.DataFrame(fetch_log)
display(fetch_frame)

# 필요하면 raw_rows[(corp_code, 2024, 'CFS')]로 전체 원본 행을 직접 확인할 수 있다.
sample_key = next((key for key, rows in raw_rows.items() if rows), None)
if sample_key:
    sample_columns = ['sj_div', 'sj_nm', 'account_id', 'account_nm', 'thstrm_amount', 'currency', 'rcept_no']
    display(pd.DataFrame(raw_rows[sample_key])[sample_columns].head(15))
    print(f'원본 확인 표본: corp_code={sample_key[0]}, year={sample_key[1]}, fs_div={sample_key[2]}')

In [ ]:
def select_contract_candidates(rows, rule):
    scoped_rows = [row for row in rows if row.get('sj_div') in rule['sj_div']]
    by_id = [row for row in scoped_rows if row.get('account_id') in rule['ids']]
    if by_id:
        return by_id, 'account_id'

    by_name = [row for row in scoped_rows if row.get('account_nm') in rule['names']]
    return (by_name, 'account_nm') if by_name else ([], None)


mapping_rows = []
for record in fetch_log:
    key = (record['corp_code'], record['사업연도'], record['재무제표 기준'])
    for label, rule in ACCOUNT_CONTRACT.items():
        candidates, matched_by = select_contract_candidates(raw_rows.get(key, []), rule)
        row = candidates[0] if len(candidates) == 1 else None
        mapping_status = '고유 매핑' if row else ('복수 후보' if candidates else '미매핑')
        mapping_rows.append({
            **record, '표준 항목': label, '매핑 상태': mapping_status, '매핑 성공': row is not None, '후보 행 수': len(candidates), '매칭 기준': matched_by,
            '후보 account_id': [candidate.get('account_id') for candidate in candidates],
            '후보 account_nm': [candidate.get('account_nm') for candidate in candidates],
            '원본 재무제표 구분': row.get('sj_div') if row else None,
            '원본 account_id': row.get('account_id') if row else None,
            '원본 account_nm': row.get('account_nm') if row else None,
            '당기 금액(원문)': row.get('thstrm_amount') if row else None,
            '통화': row.get('currency') if row else None,
            'API 단위 필드': (row.get('unit') or row.get('unit_nm') or row.get('amount_unit')) if row else None,
            '접수번호': row.get('rcept_no') if row else None,
        })

mapping_frame = pd.DataFrame(mapping_rows)
display(mapping_frame[['기업명', '사업연도', '재무제표 기준', '표준 항목', '매핑 상태', '후보 행 수', '매칭 기준', '후보 account_id', '후보 account_nm', '당기 금액(원문)', '통화', 'API 단위 필드']])

In [ ]:
# CFS와 OFS를 섞어 쓰지 않고, 기준별로 필요한 9개 항목의 완결성을 비교한다.
# 이 표는 'CFS 우선/OFS 대체'의 최종 구현 규칙을 정하는 근거이며, 아직 자동 선택을 하지 않는다.
coverage = (
    mapping_frame.groupby(['기업명', 'corp_code', '사업연도', '재무제표 기준'], dropna=False)
    .agg(매핑_항목_수=('매핑 성공', 'sum'), 필요한_항목_수=('표준 항목', 'size'), 오류=('오류', 'first'))
    .reset_index()
)
coverage['매핑 완결'] = coverage['매핑_항목_수'] == coverage['필요한_항목_수']
display(coverage)

unresolved = mapping_frame[~mapping_frame['매핑 성공']][['기업명', '사업연도', '재무제표 기준', '표준 항목', '매핑 상태', '후보 행 수', '후보 account_id', '후보 account_nm', '오류']]
if unresolved.empty:
    print('표본의 CFS/OFS에서 후보 매핑으로 고유하게 찾지 못한 표준 항목이 없습니다.')
else:
    print('다음 항목은 미매핑 또는 복수 후보입니다. 계정 ID·명칭을 원본 행에서 확인해야 합니다.')
    display(unresolved)

unit_fields = sorted({field for rows in raw_rows.values() for row in rows for field in row if 'unit' in field.lower()})
print(f'원본 행에서 확인된 단위 관련 필드: {unit_fields or "없음"}')

print('다음 판단은 실행 결과를 보고 결정합니다: CFS가 불완결할 때 OFS 전체 대체가 가능한지, 계정별 대체를 금지할지, 추가 계정 규칙이 필요한지.')

## 실행 후 볼 것

- `상태=오류`면 OpenDART가 해당 기준·연도의 데이터를 제공하지 않는 이유를 기록한다. 이것은 실패를 숨길 문제가 아니라 V1의 제한 처리 규칙 후보가 된다.
- CFS와 OFS가 모두 완결이라도 숫자를 섞어 사용하지 않는다. 최종 선택 규칙은 다음 단계에서 결과를 보고 정한다.
- `매핑 상태=복수 후보`는 임의로 첫 행을 선택하지 않는다. 계정별 의미를 확인해 최종 규칙을 정해야 한다.
- `매칭 기준=account_nm`만 반복되면 계정명 변형 위험이 있으므로, `account_id` 후보를 보강해야 한다.
- 공식 응답 가이드는 `currency`는 정의하지만 금액 단위 필드를 별도로 정의하지 않는다. 실행 결과의 `원본 행에서 확인된 단위 관련 필드`를 보고, 최종 표시 단위 규칙을 별도 데이터 계약으로 정한다.
- 이번 출력은 계정·통화·단위 필드 존재 여부·접수번호의 데이터 계약 근거다. 아직 재무비율을 계산하거나 LLM에게 해석을 맡기지 않는다.
